## 1. Merge three files to a new one, named Participation:
## a. AIM
## b. One-Source
## c. Master

In [ ]:
import pandas as pd

In [ ]:
#upload files to Colab, then read files as dataframes
df_AIM = pd.read_excel("/content/AIM NEW ITEM STORE LIST AS OF 08.14.xlsx")
df_OnceSource = pd.read_excel("/content/One Source Store List 8.26.xlsx")
df_Master = pd.read_excel("AWG All Store List 10.1.xlsx")

In [ ]:
#merge three dataframes, use left join, Master as the left table, the rest are right tables
merged_Master_AIM=pd.merge(
    left=df_Master,
    right=df_AIM,
    on='Store_Code',
    how='left')

Participation=pd.merge(
    left=df_Master_AIM,
    right=df_OneSource,
    on='Store_Code',
    how='left')

In [ ]:
#export Participation to csv format
Participation.to_csv("/content/Participation.csv", index = False)

## 2. Merge three files to a new one, named Sales_Data_Joined:
## a. the file with launch year/month, bsp, amap
## b. sales raw data
## c. Participation

In [ ]:
#upload files to Colab, then read files as dataframes
df_Sales = pd.read_excel("/content/Sales_Data_Raw.xlsx")
df_month = pd.read_excel("/content/all_month.xlsx")
df_Participation = pd.read_csv("/content/Participation.csv")

In [ ]:
#merge three dataframes, use inner join
merged_Sales_Participation=pd.merge(
    left=df_Sales,
    right=df_Participation,
    on='Store_Code',
    how='inner')

merged_Sales_Participation_month=pd.merge(
    left=merged_Sales_Participation,
    right=df_month,
    on='Item_Code',
    how='inner')

In [ ]:
#export merged_Sales_Participation_BSP to csv format
merged_Sales_Participation_month.to_csv("/content/Sales_Data_Joined.csv", index = False)

## 3. Merge two files to a new one, named Sales_Brands_Joined:
## a. Sales_Data_Joined
## b. Brands_Categories


In [ ]:
##upload files to Colab, then read files as dataframes
df_Sales_Joined = pd.read_csv("/content/Sales_Data_Joined.csv")
df_Brands = pd.read_excel("/content/Brands_Categories.xlsx")

Since the data is too large, we first drop duplicates for Brands, then use chunksize to merge.

In [ ]:
#Since data is too large, we first drop duplicates in Brands, then use chunksize to merge.
# We check the duplicates in Brands
dupes = brand[brand.duplicated(subset=[key], keep=False)]
if not dupes.empty:
    print(f"Number of duplicates in Item_Code: {dupes[key].nunique()}，Number of duplicates rows: {len(dupes)}")
    print("Duplicates Examles：")
    print(dupes.head(10))
else:
    print("There's no duplicate in Brands_Categories.")

#keep the first row for each unique Item_Code
df_Brands_unique = df_Brands.drop_duplicates(subset=[key], keep="first")

#use chunksize to merge df_Sales_Joined and df_Brands_unique
chunk_size = 50000
results = []

for start in range(0, len(merged_Sales_Participation_BSP), chunk_size):
    chunk = merged_Sales_Data_Joined.iloc[start:start + chunk_size]
    merged_chunk = chunk.merge(df_Brands_unique, on='Item_Code', how='inner')
    results.append(merged_chunk)

Sales_Brands_Joined = pd.concat(results, ignore_index=True)


In [ ]:
#export Sales_Brands_Joined to csv format
Sales_Brands_Joined.to_csv("/content/Sales_Brands_Joined.csv", index = False)

## 4. Calculate Metrics

### a. calculate the total sales and quantities for each store in 3 month after launched

In [ ]:
#For convenience, we change the name of Sales_Brands_Joined to df
#upload the file to Colab, then read the file as dataframe
df = pd.read_csv("/content/Sales_Brands_Joined.csv", low_memory=False)

#select columns we need
cols_needed = [
    "Store_Code",
    "Launch_Month",
    "Launch_Year",
    "Fiscal Year",
    "Calendar Month Nbr",
    "Billed Quantity",
    "Extended AWG Sell",
]
df = df[cols_needed].copy()

#Clean the data
#Ensure these columns are numeric format
for col in ["Launch_Month", "Launch_Year", "Fiscal Year", "Calendar Month Nbr"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Billed Quantity"] = pd.to_numeric(df["Billed Quantity"], errors="coerce").fillna(0)

#Clean all currency format
sell = df["Extended AWG Sell"].astype(str).str.strip()
sell = sell.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
sell = sell.str.replace("$", "", regex=False).str.replace(",", "", regex=False)
df["Extended AWG Sell"] = pd.to_numeric(sell, errors="coerce").fillna(0.0)

#Within the same fiscal year, identify what are the three months after items launched, generate the 3-months window
#Eg.launch month is 1, then 3-months window is 2,3,4
mask = (
    (df["Fiscal Year"] == df["Launch_Year"])
& (df["Calendar Month Nbr"] > df["Launch_Month"])
& (df["Calendar Month Nbr"] <= df["Launch_Month"] + 3)
& (df["Calendar Month Nbr"] <= 12)
)

df_3m = df.loc[mask].copy()

#Generate the string of 3-months window
def make_window(m):
    if pd.isna(m):
        return ""
    return ",".join(str(int(x)) for x in range(int(m) + 1, min(int(m) + 4, 13)))

df_3m["Months Window"] = df_3m["Launch_Month"].apply(make_window)

#Calculate sums of Billed Quantity and Extended AWG Sell within each window
result = (
    df_3m.groupby(["Store_Code", "Launch_Year", "Launch_Month", "Months Window"], as_index=False)
    .agg({
        "Billed Quantity": "sum",
        "Extended AWG Sell": "sum"
    })
    .rename(columns={
        "Billed Quantity": "Billed Quantity_3M",
        "Extended AWG Sell": "Extended AWG Sell_3M"
    })
)

result.to_csv("result.csv", index=False, encoding="utf-8-sig")

print(result.head())

### b. calculate distribution of stores which sell items in 3 months after items launched

In [ ]:
from pathlib import Path

file_path = "/content/Sales_Brands_Joined.csv"
output_path = "/content/Item_Store_Coverage_3M_with_window.csv"

use_cols = [
    "Item_Code", "Store_Code", "Launch_Month", "Launch_Year", "Fiscal Year",
    "Calendar Month Nbr", "Billed Quantity"
]
df = pd.read_csv(file_path, usecols=use_cols, low_memory=False, dtype=str)

#Clean the data
#Ensure these columns are numeric format
for c in ["Launch_Month", "Launch_Year", "Fiscal Year", "Calendar Month Nbr"]:
  df[c] = pd.to_numeric(df[c], errors="coerce")
df["Billed Quantity"] = pd.to_numeric(df["Billed Quantity"], errors="coerce").fillna(0)

#Within the same fiscal year, identify what are the three months after items launched, generate the 3-months window
#Eg.launch month is 1, then 3-months window is 2,3,4
mask = (
    (df["Fiscal Year"] == df["Launch_Year"]) &
    (df["Calendar Month Nbr"] > df["Launch_Month"]) &
    (df["Calendar Month Nbr"] <= df["Launch_Month"] + 3) &
    (df["Calendar Month Nbr"] <= 12)
)
#Only keep rows within each window that items have been sold
df_active = df.loc[mask & (df["Billed Quantity"] > 0)].copy()

#Display 3-months windows
def months_window_str(launch_m: float) -> str:
  if pd.isna(launch_m):
      return ""
    m = int(launch_m)
    months = [x for x in (m+1, m+2, m+3) if 1 <= x <= 12]
    return ",".join(map(str, months))
df_active["Months Window"] = df_active["Launch_Month"].apply(months_window_str)

#Calculate the unique number of Store_Code
total_stores = df["Store_Code"].nunique()

#Calculate the distribution for each item, within each fiscal year and launch month
result2 = (
    df_active
    .groupby(["Item_Code","Launch_Year","Fiscal Year","Launch_Month","Months Window"], as_index=False)["Store_Code"]
    .nunique()
    .rename(columns={"Store_Code":"Active Stores"})
)

#export the result
result2["Total Stores"] = total_stores
result2["%Stores_in_3M"] = result2["Active Stores"] / result2["Total Stores"] * 100

Path(output_path).parent.mkdir(parents=True, exist_ok=True)
result2.to_csv(output_path, index=False, encoding="utf-8-sig")
print("completed：", output_path)
print(result2.head())


## 5. Merge Sales_Brands_Joined & Item_Store_Coverage_3M_with_window, named Sales_Brands_3M

In [ ]:
##upload the file to Colab, then read the file as dataframe
main_path   = "/content/Sales_Brands_Joined.csv"
post_path   = "/content/Item_Store_Coverage_3M_with_window.csv"
output_path = "/content/Sales_Brands_3M.csv"

main = pd.read_csv(main_path, dtype=str)
post = pd.read_csv(post_path, dtype=str)
keys = ["Store_Code", "Launch_Year", "Launch_Month"]
merged = main.merge(post, on=keys, how="left")

#export the result
merged.to_csv(output_path, index=False, encoding="utf-8-sig")